# Reduced-Persona Timeseries Analysis
Diagnostic notebook for comparing max/min survival seeds and detecting collapse patterns.

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy import signal, fft
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load summary and identify max/min survival seeds
summary_path = Path('/home/user/personality-dungeon/outputs/actionD_reduced_persona_scan/reduced_persona_scan_summary.json')
with open(summary_path) as f:
    summary = json.load(f)

# Extract reduced3 rows
rows = summary['rows']
reduced3_rows = [r for r in rows if r['condition'] == 'reduced3']

# Find max/min dropout_round
valid_dropouts = [(r['seed'], r['dropout_round']) for r in reduced3_rows if r['dropout_round'] is not None]
valid_dropouts.sort(key=lambda x: x[1], reverse=True)

print(f'Reduced-3 seeds with valid dropout times:')
for seed, dropout in valid_dropouts:
    print(f'  seed {seed}: dropout_round={dropout}')

max_seed, max_dropout = valid_dropouts[0] if valid_dropouts else (None, None)
min_seed, min_dropout = valid_dropouts[-1] if valid_dropouts else (None, None)

print(f'\nMax survival: seed {max_seed} (dropout={max_dropout})')
print(f'Min survival: seed {min_seed} (dropout={min_dropout})')

In [ ]:
# Load CSV data for max/min seeds
def load_seed_csv(seed, condition):
    if condition == 'reduced3':
        csv_path = f'/home/user/personality-dungeon/outputs/actionD_reduced_persona_scan/reduced3_assertiveness_stability_seeking_curiosity/seed{seed}.csv'
    else:
        csv_path = f'/home/user/personality-dungeon/outputs/actionD_reduced_persona_scan/control/seed{seed}.csv'
    return pd.read_csv(csv_path)

df_max = load_seed_csv(max_seed, 'reduced3')
df_min = load_seed_csv(min_seed, 'reduced3')

print(f'Loaded CSV for seed {max_seed}: {len(df_max)} rounds')
print(f'Loaded CSV for seed {min_seed}: {len(df_min)} rounds')
print(f'\nColumns: {list(df_max.columns)[:10]}...')

In [ ]:
# Plot strategy proportions for both seeds
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Max seed: strategy proportions
ax = axes[0, 0]
ax.plot(df_max['round'], df_max['p_aggressive'], label='aggressive', alpha=0.7)
ax.plot(df_max['round'], df_max['p_defensive'], label='defensive', alpha=0.7)
ax.plot(df_max['round'], df_max['p_balanced'], label='balanced', alpha=0.7)
ax.axvline(max_dropout, color='red', linestyle='--', alpha=0.5, label=f'dropout={max_dropout}')
ax.set_title(f'Max Survival: seed {max_seed} (dropout={max_dropout})')
ax.set_ylabel('Strategy Proportion')
ax.legend(loc='best')
ax.grid(alpha=0.3)

# Min seed: strategy proportions
ax = axes[1, 0]
ax.plot(df_min['round'], df_min['p_aggressive'], label='aggressive', alpha=0.7)
ax.plot(df_min['round'], df_min['p_defensive'], label='defensive', alpha=0.7)
ax.plot(df_min['round'], df_min['p_balanced'], label='balanced', alpha=0.7)
ax.axvline(min_dropout, color='red', linestyle='--', alpha=0.5, label=f'dropout={min_dropout}')
ax.set_title(f'Min Survival: seed {min_seed} (dropout={min_dropout})')
ax.set_ylabel('Strategy Proportion')
ax.set_xlabel('Round')
ax.legend(loc='best')
ax.grid(alpha=0.3)

# Amplitude comparison
ax = axes[0, 1]
max_amp = np.sqrt(df_max['p_aggressive']**2 + df_max['p_defensive']**2 + df_max['p_balanced']**2)
min_amp = np.sqrt(df_min['p_aggressive']**2 + df_min['p_defensive']**2 + df_min['p_balanced']**2)
ax.plot(df_max['round'], max_amp, label=f'seed {max_seed}', alpha=0.7)
ax.plot(df_min['round'], min_amp, label=f'seed {min_seed}', alpha=0.7)
ax.set_title('System Amplitude (L2 norm)')
ax.set_ylabel('Amplitude')
ax.legend()
ax.grid(alpha=0.3)

# Entropy-like measure (neg-sum-p*log(p))
ax = axes[1, 1]
def entropy(p_a, p_d, p_b):
    eps = 1e-9
    return -p_a*np.log(p_a+eps) - p_d*np.log(p_d+eps) - p_b*np.log(p_b+eps)

max_ent = entropy(df_max['p_aggressive'], df_max['p_defensive'], df_max['p_balanced'])
min_ent = entropy(df_min['p_aggressive'], df_min['p_defensive'], df_min['p_balanced'])
ax.plot(df_max['round'], max_ent, label=f'seed {max_seed}', alpha=0.7)
ax.plot(df_min['round'], min_ent, label=f'seed {min_seed}', alpha=0.7)
ax.set_title('System Entropy (higher=more mixed)')
ax.set_ylabel('Entropy')
ax.set_xlabel('Round')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/home/user/personality-dungeon/outputs/actionD_reduced_persona_scan/timeseries_comparison.png', dpi=100)
plt.show()
print('Saved: timeseries_comparison.png')

In [ ]:
# Analyze collapse behavior
# Compute volatility (rolling std of strategy deltas)
def rolling_volatility(df, window=100):
    # Pick p_aggressive as proxy
    p_a = df['p_aggressive'].values
    volatility = np.array([np.std(p_a[max(0, i-window):i]) for i in range(len(p_a))])
    return volatility

max_vol = rolling_volatility(df_max, window=100)
min_vol = rolling_volatility(df_min, window=100)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Max seed volatility
ax = axes[0]
ax.plot(df_max['round'], max_vol, linewidth=1, alpha=0.7, color='blue')
ax.axvline(max_dropout, color='red', linestyle='--', alpha=0.5, label=f'dropout={max_dropout}')
ax.set_title(f'Volatility: Max Survival (seed {max_seed})')
ax.set_xlabel('Round')
ax.set_ylabel('Rolling Std (window=100)')
ax.legend()
ax.grid(alpha=0.3)

# Min seed volatility
ax = axes[1]
ax.plot(df_min['round'], min_vol, linewidth=1, alpha=0.7, color='orange')
ax.axvline(min_dropout, color='red', linestyle='--', alpha=0.5, label=f'dropout={min_dropout}')
ax.set_title(f'Volatility: Min Survival (seed {min_seed})')
ax.set_xlabel('Round')
ax.set_ylabel('Rolling Std (window=100)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/home/user/personality-dungeon/outputs/actionD_reduced_persona_scan/volatility_comparison.png', dpi=100)
plt.show()
print('Saved: volatility_comparison.png')

# Print collapse signatures
print(f'\nMax seed (seed {max_seed}, dropout={max_dropout})')
print(f'  Pre-dropout volatility (rounds {max(0, max_dropout-200)}:{max_dropout}): {np.mean(max_vol[max(0, max_dropout-200):max_dropout]):.6f}')
print(f'\nMin seed (seed {min_seed}, dropout={min_dropout})')
print(f'  Pre-dropout volatility (rounds {max(0, min_dropout-200)}:{min_dropout}): {np.mean(min_vol[max(0, min_dropout-200):min_dropout]):.6f}')

In [ ]:
# FFT analysis to detect noise vs depletion
# Depletion: slow decline (low freq) | Noise burst: high freq spikes

def fft_analysis(signal_arr, dropout_round=None):
    # Trim to dropout if available
    if dropout_round is not None:
        signal_arr = signal_arr[:dropout_round]
    
    # Detrend and window
    signal_arr = signal.detrend(signal_arr)
    window = signal.windows.hann(len(signal_arr))
    signal_windowed = signal_arr * window
    
    # FFT
    freqs = np.fft.fftfreq(len(signal_windowed))
    fft_vals = np.abs(np.fft.fft(signal_windowed)) ** 2
    
    # Energy in low vs high frequencies
    n = len(freqs)
    low_freq_energy = np.sum(fft_vals[:n//4])  # 0-0.25 cycles
    high_freq_energy = np.sum(fft_vals[n//4:])  # 0.25-1 cycles
    
    return freqs[:n//2], fft_vals[:n//2], low_freq_energy, high_freq_energy

max_freqs, max_fft, max_low, max_high = fft_analysis(df_max['p_aggressive'].values, max_dropout)
min_freqs, min_fft, min_low, min_high = fft_analysis(df_min['p_aggressive'].values, min_dropout)

print(f'Max seed (seed {max_seed}): Low-freq energy={max_low:.2e}, High-freq energy={max_high:.2e}')
print(f'  Ratio (low/high)={max_low/(max_high+1e-10):.3f}')
print(f'\nMin seed (seed {min_seed}): Low-freq energy={min_low:.2e}, High-freq energy={min_high:.2e}')
print(f'  Ratio (low/high)={min_low/(min_high+1e-10):.3f}')

# Plot power spectra
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.semilogy(max_freqs, max_fft, linewidth=1, alpha=0.7, color='blue')
ax.set_title(f'Power Spectrum: Max Survival (seed {max_seed})')
ax.set_xlabel('Frequency')
ax.set_ylabel('Power')
ax.grid(alpha=0.3, which='both')

ax = axes[1]
ax.semilogy(min_freqs, min_fft, linewidth=1, alpha=0.7, color='orange')
ax.set_title(f'Power Spectrum: Min Survival (seed {min_seed})')
ax.set_xlabel('Frequency')
ax.set_ylabel('Power')
ax.grid(alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('/home/user/personality-dungeon/outputs/actionD_reduced_persona_scan/fft_comparison.png', dpi=100)
plt.show()
print('Saved: fft_comparison.png')

In [ ]:
# Determine collapse signature
print('='*60)
print('COLLAPSE SIGNATURE ANALYSIS')
print('='*60)

max_ratio = max_low / (max_high + 1e-10)
min_ratio = min_low / (min_high + 1e-10)

print(f'\nMax Survival (seed {max_seed}, dropout={max_dropout})')
print(f'  Low-freq / High-freq ratio: {max_ratio:.3f}')
if max_ratio > 1.0:
    print('  -> Dominated by low-frequency energy: ENERGY DEPLETION pattern')
else:
    print('  -> Dominated by high-frequency energy: NOISE BURST pattern')

print(f'\nMin Survival (seed {min_seed}, dropout={min_dropout})')
print(f'  Low-freq / High-freq ratio: {min_ratio:.3f}')
if min_ratio > 1.0:
    print('  -> Dominated by low-frequency energy: ENERGY DEPLETION pattern')
else:
    print('  -> Dominated by high-frequency energy: NOISE BURST pattern')

print(f'\nDifference in dropout timing: {abs(max_dropout - min_dropout)} rounds')
print(f'Difference in frequency signature: {abs(max_ratio - min_ratio):.3f}')

if max_dropout == min_dropout:
    print('\n*** KEY FINDING: Both seeds collapse at nearly same time despite different frequencies')
    print('    -> Suggests system has hard limit (not individual seed variance)')
else:
    survival_diff_pct = 100 * abs(max_dropout - min_dropout) / max_dropout
    print(f'\n*** Survival variation: {survival_diff_pct:.1f}% difference')
    if survival_diff_pct < 5:
        print('    -> Variation is minimal: suggests deterministic collapse mechanism')
    else:
        print('    -> Moderate variation: suggests seed-dependent randomness')

print('\n' + '='*60)

In [ ]:
# Prepare SDD record
sdd_finding = f"""
## W3.6 - Reduced-Persona Empirical Verdict (actionD)

**Hypothesis**: Reducing 9-persona to 3-core traits (assertiveness, stability_seeking, curiosity) would stabilize B1/B2 by removing noise.

**Scope**: 6 seeds × 6000 rounds, memory_kernel=3, players=300

**Outcome (NEGATIVE)**:
- Control baseline dropout_median=4820 short_s3_mean=0.5334
- Reduced-3 dropout_median=4820 short_s3_mean=0.5334 (shift=0)
- No measurable lifespan extension; short_l3_count=0 (relight gate failed)

**Collapse Signature Analysis**:
- Max-survival seed {max_seed}: dropout={max_dropout}, low/high freq ratio={max_ratio:.3f}
- Min-survival seed {min_seed}: dropout={min_dropout}, low/high freq ratio={min_ratio:.3f}
- Collapse pattern: {'ENERGY DEPLETION' if max_ratio > 1 else 'NOISE BURST'}
- Variation: {abs(max_dropout-min_dropout)} rounds ({100*abs(max_dropout-min_dropout)/max_dropout:.1f}%)

**Implication**:
1. Persona dimensionality is NOT the root cause of B1/B2 instability.
2. The collapse appears driven by {"intrinsic payoff geometry" if max_ratio > 1 else "extrinsic stochastic events"}.
3. Both geometry interventions (Option2, Option3) and dimensionality reduction (Option4) have failed:
   - Pulses (Option2) did not extend lifespan sufficiently (+200 gate).
   - Projections (Option3) showed ephemeral signals; 10k confirm was negative.
   - Reduced-persona (Option4) had zero effect on dropout timing.

**Next Path**: Structural intervention required—either (a) investigate payoff matrix invariants, or (b) test hybrid persona+geometry combinations.
"""

print(sdd_finding)
print(f"\n\n📎 SDD Record Template (paste into SDD.md):")
print(sdd_finding)